# Chapter 126: Mamba for Trading

This notebook demonstrates how to use the Mamba architecture for trading signal prediction.

## Contents
1. Data Loading (Stock and Crypto)
2. Feature Engineering
3. Model Training
4. Backtesting
5. Results Analysis

In [ ]:
# Import libraries
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from mamba_model import MambaTradingModel, create_mamba_trading_model
from data_loader import YahooFinanceLoader, BybitDataLoader, DataManager
from features import FeatureEngineer, create_labels, prepare_sequences, normalize_features
from train import MambaTrainer, TrainingConfig, create_data_loaders, train_mamba_model
from backtest import MambaBacktest, print_backtest_summary, plot_backtest_results

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Data Loading

We'll load data from two sources:
- Stock market data from Yahoo Finance
- Cryptocurrency data from Bybit

In [ ]:
# Initialize data manager
data_manager = DataManager()

# Load stock data (Apple)
print("Loading stock data...")
stock_df = data_manager.get_stock_data("AAPL", period="2y")
print(f"Stock data shape: {stock_df.shape}")
print(stock_df.head())

In [ ]:
# Load cryptocurrency data (Bitcoin)
print("\nLoading crypto data from Bybit...")
try:
    crypto_df = data_manager.get_crypto_data("BTCUSDT", interval="60", days=30)
    print(f"Crypto data shape: {crypto_df.shape}")
    print(crypto_df.head())
except Exception as e:
    print(f"Could not load crypto data: {e}")
    crypto_df = None

In [ ]:
# Visualize price data
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Stock price
axes[0].plot(stock_df.index, stock_df['close'], 'b-', linewidth=1)
axes[0].set_title('AAPL Stock Price')
axes[0].set_ylabel('Price ($)')
axes[0].grid(True, alpha=0.3)

# Crypto price (if available)
if crypto_df is not None:
    axes[1].plot(crypto_df.index, crypto_df['close'], 'orange', linewidth=1)
    axes[1].set_title('BTC/USDT Price (Bybit)')
    axes[1].set_ylabel('Price ($)')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Feature Engineering

Create trading-relevant features from the raw OHLCV data.

In [ ]:
# Initialize feature engineer
fe = FeatureEngineer(include_volume=True)

# Compute features for stock data
stock_features = fe.compute_all_features(stock_df)
print(f"Total features computed: {len(stock_features.columns)}")
print(f"\nFeature columns:")
print(stock_features.columns.tolist())

In [ ]:
# Create labels for classification (buy/hold/sell)
labels = create_labels(
    stock_df,
    method='triple_barrier',
    threshold=0.02,  # 2% threshold for buy/sell
    forward_periods=5  # 5-day forward returns
)

# Remove NaN labels
labels = labels.dropna()
stock_features = stock_features.loc[labels.index]

print(f"Label distribution:")
print(labels.value_counts().sort_index())
print(f"\n0 = Sell, 1 = Hold, 2 = Buy")

In [ ]:
# Prepare sequences for the model
LOOKBACK = 60  # Use 60 days of history

# Select feature columns (exclude original OHLCV)
feature_cols = [
    'returns', 'returns_5', 'returns_10', 'returns_20',
    'log_returns', 'sma_5', 'sma_10', 'sma_20', 'sma_50',
    'close_sma_5_ratio', 'close_sma_20_ratio',
    'macd', 'macd_signal', 'macd_hist',
    'volatility_10', 'volatility_20', 'atr_14',
    'bb_width', 'bb_position',
    'rsi_14', 'stoch_k', 'stoch_d',
    'roc_10', 'momentum_10',
    'volume_ratio_10', 'obv', 'mfi_14'
]

# Filter to available columns
feature_cols = [c for c in feature_cols if c in stock_features.columns]
print(f"Using {len(feature_cols)} features")

X, y = prepare_sequences(
    stock_features,
    labels,
    lookback=LOOKBACK,
    feature_columns=feature_cols
)

print(f"\nSequence shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
# Split data (time-series aware: no shuffling, chronological split)
train_size = int(len(X) * 0.7)
val_size = int(len(X) * 0.15)

X_train = X[:train_size]
y_train = y[:train_size]
X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]
X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print(f"Train set: {X_train.shape[0]} samples")
print(f"Val set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# Normalize features
X_train_norm, X_val_norm, stats = normalize_features(X_train, X_val)
_, X_test_norm, _ = normalize_features(X_train, X_test)

print("Features normalized using z-score normalization")

## 3. Model Training

Train the Mamba model for trading signal classification.

In [ ]:
# Create Mamba model
n_features = X_train.shape[2]

model = create_mamba_trading_model(
    n_features=n_features,
    preset='default',  # 'small', 'default', or 'large'
    n_classes=3,  # sell, hold, buy
    task='classification'
)

print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")
print(model)

In [ ]:
# Configure training
config = TrainingConfig(
    learning_rate=1e-3,
    batch_size=32,
    epochs=50,
    early_stopping_patience=10,
    weight_decay=0.01,
)

# Train model
print("Training Mamba model...")
model, history = train_mamba_model(
    model,
    X_train_norm, y_train.astype(np.int64),
    X_val_norm, y_val.astype(np.int64),
    config=config,
    verbose=True
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Validation')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Backtesting

Evaluate the model's trading performance on the test set.

In [ ]:
# Prepare test data for backtesting
test_start_idx = train_size + val_size + LOOKBACK
test_df = stock_df.iloc[test_start_idx:test_start_idx + len(X_test)].copy()

# Convert test features to tensor
X_test_tensor = torch.FloatTensor(X_test_norm)

print(f"Test period: {test_df.index[0]} to {test_df.index[-1]}")
print(f"Test samples: {len(test_df)}")

In [ ]:
# Run backtest
backtest = MambaBacktest(
    model=model,
    initial_capital=100000,
    position_size=1.0,
    transaction_cost=0.001,
    allow_short=False
)

result = backtest.run(
    test_df,
    X_test_tensor,
    buy_threshold=0.5,
    sell_threshold=0.5,
    stop_loss=0.05,  # 5% stop loss
    take_profit=0.10  # 10% take profit
)

# Print summary
print_backtest_summary(result)

In [ ]:
# Plot backtest results
plot_backtest_results(result)

## 5. Results Analysis

Analyze the model's predictions and trading performance.

In [ ]:
# Get model predictions on test set
model.eval()
with torch.no_grad():
    probs = model.predict_proba(X_test_tensor)
    predictions = probs.argmax(dim=1).numpy()

# Classification report
from collections import Counter
print("Prediction distribution:")
print(Counter(predictions))
print("\n0 = Sell, 1 = Hold, 2 = Buy")

In [ ]:
# Compare with buy-and-hold strategy
buy_hold_return = (test_df['close'].iloc[-1] / test_df['close'].iloc[0] - 1) * 100

print(f"\nStrategy Comparison:")
print(f"Mamba Strategy Return: {result.total_return:.2f}%")
print(f"Buy-and-Hold Return: {buy_hold_return:.2f}%")
print(f"Outperformance: {result.total_return - buy_hold_return:.2f}%")

In [ ]:
# Visualize predictions over time
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price with buy/sell signals
ax1 = axes[0]
ax1.plot(test_df.index[:len(predictions)], test_df['close'].iloc[:len(predictions)], 'b-', alpha=0.7)

buy_signals = predictions == 2
sell_signals = predictions == 0

ax1.scatter(test_df.index[:len(predictions)][buy_signals], 
            test_df['close'].iloc[:len(predictions)][buy_signals],
            marker='^', color='green', s=50, label='Buy Signal')
ax1.scatter(test_df.index[:len(predictions)][sell_signals],
            test_df['close'].iloc[:len(predictions)][sell_signals],
            marker='v', color='red', s=50, label='Sell Signal')
ax1.set_ylabel('Price ($)')
ax1.set_title('Price with Trading Signals')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Prediction probabilities
ax2 = axes[1]
probs_np = probs.numpy()
ax2.fill_between(range(len(probs_np)), probs_np[:, 2], alpha=0.5, color='green', label='Buy Prob')
ax2.fill_between(range(len(probs_np)), -probs_np[:, 0], alpha=0.5, color='red', label='Sell Prob')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Probability')
ax2.set_title('Model Confidence')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Equity curve comparison
ax3 = axes[2]
ax3.plot(result.equity_curve, 'b-', label='Mamba Strategy')
buy_hold_equity = 100000 * (test_df['close'].iloc[:len(result.equity_curve)] / test_df['close'].iloc[0])
ax3.plot(buy_hold_equity.values, 'gray', alpha=0.7, label='Buy & Hold')
ax3.set_xlabel('Time Step')
ax3.set_ylabel('Portfolio Value ($)')
ax3.set_title('Equity Curve Comparison')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
torch.save(model.state_dict(), 'mamba_trading_model.pt')
print("Model saved to mamba_trading_model.pt")

## Summary

In this notebook, we demonstrated:

1. **Data Loading**: Fetching stock data from Yahoo Finance and crypto data from Bybit
2. **Feature Engineering**: Creating technical indicators and trading features
3. **Model Training**: Training a Mamba model for trading signal classification
4. **Backtesting**: Evaluating the strategy with realistic transaction costs
5. **Analysis**: Comparing performance against buy-and-hold benchmark

### Key Takeaways

- Mamba's selective state space mechanism is well-suited for capturing long-range dependencies in financial time series
- The linear complexity allows efficient processing of long historical sequences
- Feature engineering remains crucial for model performance
- Always validate with realistic backtesting including transaction costs

### Next Steps

- Experiment with different lookback periods
- Try the model on cryptocurrency data
- Implement more sophisticated position sizing
- Add risk management rules
- Test on out-of-sample data